# VCC 2026 data explorer

In [ ]:
import sys, json, math, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, h5py, anndata as ad, scipy.sparse as sp
import matplotlib.pyplot as plt
from scipy.special import expit
from IPython.display import display, Markdown
try:
    from anndata.io import read_elem
except ImportError:
    from anndata.experimental import read_elem
warnings.filterwarnings('ignore', category=RuntimeWarning)

V = Path.home() / 'vcc'
DATA, LIB = V / 'data', V / 'lib'
WORK = LIB / 'review_v4'                      # the pipeline's working folder (read-only here)
CACHE = LIB / 'explore'; CACHE.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(V / 'code'))            # reuse the pipeline's own helpers
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.precision', 4)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25})

GENES = pd.read_csv(DATA / 'gene_names.csv')['gene_name'].astype(str).to_numpy()
GENE_POS = {g: i for i, g in enumerate(GENES)}
PANEL = pd.read_csv(DATA / 'pert_counts.csv')['target_gene'].astype(str).to_numpy()
CONTROL = 'non-targeting'
PC = 0.5                                       # CPM pseudocount for exploratory log fold changes
MIN_CELLS = 20                                 # same minimum the pipeline uses; smaller rows are mostly noise

# key -> (file, storage kind, biological background)
SOURCES = {
    'HCT116':     (LIB / 'orion/HCT116_pseudobulk.h5ad',      'sums',    'HCT116'),
    'HEK293T':    (LIB / 'orion/HEK293T_pseudobulk.h5ad',     'sums',    'HEK293T'),
    'CD4_Stim48': (LIB / 'tcell/TCELL_Stim48hr_effects.h5ad', 'effects', 'CD4_T'),
    'CD4_Rest':   (LIB / 'tcell/TCELL_Rest_effects.h5ad',     'effects', 'CD4_T'),
    'H1':         (LIB / 'h1/H1_full_2025.h5ad',              'cells',   'H1'),
}
LINES = {k: (DATA / f'context_{k}.h5ad', 'cells', f'line {k}') for k in 'ABC'}
ALL = {**SOURCES, **LINES}
missing = [str(p) for p, _, _ in ALL.values() if not p.exists()]
print(f'{len(GENES):,} genes | {len(PANEL)} challenge targets | cache: {CACHE}')
print('missing files:', missing or 'none')

## 1. Header and footer of every file

Each `.h5ad` holds a cell (or row) table `obs`, a gene table `var`, a matrix `X`, and optionally extra matrices in `layers`. Files are opened in **backed** mode, so only the rows shown are read from disk.

What each file stores:

| Context | Rows are | `X` / layers |
|---|---|---|
| HCT116, HEK293T | one row per knockdown, plus one aggregated control row | layers `sum_counts`, `sum_cpm`, `sum_cpm2`, `n_expr`: per-gene sums over that row's cells |
| CD4_T Stim48hr / Rest | one row per knockdown | `X` = log2 fold change vs control; layer `se`; control CPM in `var['ctrl_cpm']` |
| H1 | one row per **cell** | `X` = raw UMI counts |
| A, B, C | one row per **control cell** | `X` = raw UMI counts |

In [ ]:
def _block(matrix, rows, cols, gene_names):
    m = matrix[rows[0]:rows[1]]
    m = m.toarray() if sp.issparse(m) else np.asarray(m)
    return pd.DataFrame(m[:, cols[0]:cols[1]], columns=gene_names[cols[0]:cols[1]])

def peek(name, n=5, k=8):
    path, kind, background = ALL[name]
    display(Markdown(f'### {name}  ·  `{path.name}`  ·  {path.stat().st_size/1e9:.2f} GB  ·  background: {background}'))
    a = ad.read_h5ad(path, backed='r')
    try:
        genes = a.var_names.astype(str).to_numpy()
        x_kind = type(a.X).__name__
        print(f'shape {a.n_obs:,} rows x {a.n_vars:,} genes | X: {x_kind} | layers: {[k for k in a.layers.keys() if k]} | uns: {list(a.uns.keys())}')
        print(f'genes also in the 2026 list: {np.isin(genes, GENES).sum():,} / {len(GENES):,}')
        display(Markdown('**obs (row table): first and last rows**')); display(pd.concat([a.obs.head(n), a.obs.tail(n)]))
        display(Markdown('**var (gene table): first and last genes**')); display(pd.concat([a.var.head(n), a.var.tail(n)]))
        mats = {'X': a.X} if kind != 'sums' else {'layers/sum_counts': a.layers['sum_counts'], 'layers/n_expr': a.layers['n_expr']}
        if kind == 'filtered': mats['layers/se'] = a.layers['se']      # log2FC and its standard error
        for label, m in mats.items():
            display(Markdown(f'**{label}: first {n} rows x first {k} genes, then last {n} rows x last {k} genes**'))
            head = _block(m, (0, n), (0, k), genes); head.index = a.obs_names[:n]
            tail = _block(m, (a.n_obs - n, a.n_obs), (a.n_vars - k, a.n_vars), genes); tail.index = a.obs_names[-n:]
            display(head); display(tail)
        for key, value in a.uns.items():
            print(f'uns[{key!r}]: {str(value)[:300]}')
    finally:
        a.file.close()

display(Markdown('### gene_names.csv and pert_counts.csv'))
display(pd.concat([pd.read_csv(DATA/'gene_names.csv').head(3), pd.read_csv(DATA/'gene_names.csv').tail(3)]))
display(pd.concat([pd.read_csv(DATA/'pert_counts.csv').head(3), pd.read_csv(DATA/'pert_counts.csv').tail(3)]))

### Filtered datasets (`lib/filters/`)

One file per training dataset, built by `scripts/knockdown_filters.py`. Rows are the knockdowns that passed all three filters (at least 100 cells, target expressed in that dataset's controls, target dropped by at least 50% with adjusted p < 0.05). Columns are the challenge genes the dataset measured. `X` is the log2 fold change vs that dataset's controls. `obs` holds each knockdown's QC numbers and `var` each gene's control expression.

In [ ]:
# Self-contained: does not depend on peek(), so it always shows both the log2FC and the SE tables.
def peek_filtered(path, n=5, k=8):
    a = ad.read_h5ad(path, backed='r')
    try:
        genes = a.var_names.astype(str).to_numpy()
        display(Markdown(f'### {path.stem}  ·  `{path.name}`  ·  {path.stat().st_size/1e9:.2f} GB'))
        print(f'{a.n_obs:,} kept knockdowns x {a.n_vars:,} genes | layers: {list(a.layers.keys())}')
        display(Markdown('**obs: one row per knockdown (first and last)**')); display(pd.concat([a.obs.head(n), a.obs.tail(n)]))
        display(Markdown('**var: one row per gene, control expression (first and last)**')); display(pd.concat([a.var.head(n), a.var.tail(n)]))
        for label, m in (('X = log2 fold change vs controls', a.X), ('layers/se = standard error of that log2FC', a.layers['se'])):
            display(Markdown(f'**{label}: first {n} knockdowns x first {k} genes, then last {n} x last {k}**'))
            head = pd.DataFrame(np.asarray(m[:n, :k]), index=a.obs_names[:n], columns=genes[:k])
            tail = pd.DataFrame(np.asarray(m[a.n_obs - n:, a.n_vars - k:]), index=a.obs_names[-n:], columns=genes[-k:])
            display(head); display(tail)
    finally:
        a.file.close()

FILTERED = sorted((LIB / 'filters').glob('*.h5ad'))
print('filtered files:', [p.name for p in FILTERED] or 'none: run scripts/knockdown_filters.py first')
for path in FILTERED:
    peek_filtered(path)


## 3. Raw single-cell files
Every perturbation source we have (or are downloading) as raw single cells, before any cleaning. Each file is opened without loading it into memory. Shown per file: cells × genes and how the counts are stored, the first 20 cells' metadata (`obs`), the first 20 genes (`var`), and raw counts for those 20 cells over the 20 genes with the most counts in them (the first genes are often all zero).

Not here: CD4 T cells (effect tables only) and the full X-Atlas/Orion release (~520 GB; we hold HCT116 / HEK293T cells for the A–C targets only). A file not yet downloaded is listed as missing.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from IPython.display import display
V = Path.home() / 'vcc'
PB = V / 'lib/downloads/perturbase'
RAW = {
    'K562':     (V / 'lib/replogle/K562_gwps_raw_singlecell_01.h5ad', 'Replogle 2022, genome-wide CRISPRi'),
    'RPE1':     (V / 'lib/downloads/rpe1/rpe1_raw_singlecell_01.h5ad', 'Replogle 2022, essential genes'),
    'HepG2':    (V / 'lib/downloads/nadig_hepg2/GSE264667_hepg2_raw_singlecell_01.h5ad', 'Nadig 2025'),
    'Jurkat':   (V / 'lib/downloads/nadig_jurkat/GSE264667_jurkat_raw_singlecell_01.h5ad', 'Nadig 2025'),
    'H1':       (V / 'lib/h1/H1_full_2025.h5ad', 'Arc 2025 (our validation line)'),
    'HCT116':   (V / 'lib/orion/HCT116_cells_targets.h5ad', 'X-Atlas/Orion 2025, A-C targets + controls only'),
    'HEK293T':  (V / 'lib/orion/HEK293T_cells_targets.h5ad', 'X-Atlas/Orion 2025, A-C targets + controls only'),
    'hESC':     (PB / 'PRJNA524331/raw.h5ad', 'Genga 2019, during endoderm differentiation'),
    'HFF':      (PB / 'CRISPRi_perturb_host/raw.h5ad', 'Hein 2021, foreskin fibroblasts; kept: the uninfected 00h cells only'),
}

def show_raw(name, n=20):
    path, note = RAW[name]
    if not path.exists():
        print(f'{name}: not on disk yet ({path})'); return
    a = ad.read_h5ad(path, backed='r')
    X = a.X
    kind = 'sparse (CSR)' if hasattr(X, 'format') or 'sparse' in type(X).__name__.lower() else 'dense'
    print(f'{name} - {note}\n{path}\n{a.n_obs:,} cells x {a.n_vars:,} genes; counts stored {kind}, {X.dtype}; {path.stat().st_size / 1e9:.1f} GB')
    print(f'obs columns: {list(a.obs.columns)}\nvar columns: {list(a.var.columns)}')
    display(a.obs.head(n)); display(a.var.head(n))
    block = X[:n]
    block = block.toarray() if sp.issparse(block) or hasattr(block, 'toarray') else np.asarray(block)
    names = a.var['gene_name'].astype(str).to_numpy() if 'gene_name' in a.var else a.var_names.astype(str).to_numpy()
    top = np.argsort(-block.sum(0), kind='stable')[:n]
    print(f'raw counts: first {n} cells x the {n} genes with the most counts in them')
    display(pd.DataFrame(block[:, top], index=a.obs_names[:n], columns=names[top]))
    a.file.close()

pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
pd.DataFrame([(k, v[1], v[0].exists(), round(v[0].stat().st_size / 1e9, 2) if v[0].exists() else None) for k, v in RAW.items()],
             columns=['dataset', 'source', 'on disk', 'GB'])

In [ ]:
show_raw('K562')

In [ ]:
show_raw('RPE1')

In [ ]:
show_raw('HepG2')

In [ ]:
show_raw('Jurkat')

In [ ]:
show_raw('H1')

In [ ]:
show_raw('HCT116')

In [ ]:
show_raw('HEK293T')

In [ ]:
show_raw('hESC')

In [ ]:
show_raw('HFF')

### 3b. How usable are the PerturBase label sets?
Per file: cells by label kind (control, single knockdown, pair, 3+ guides, no guide), split at 1,000 UMIs to separate real cells from near-empty droplets, and how many single knockdowns reach **30 cells** (our QC). HFF and hESC are already cell-filtered. (The Tian 2019 / 2021 iPSC and neuron screens were reviewed and removed on 2026-10-18: most knockdowns lowered their target by less than 50%, and Tian 2019 was unfiltered droplet output.) Note that three files store counts column by column (CSC); the UMI sums below handle both layouts.

In [ ]:
import h5py
from anndata.io import read_elem

def cell_umis(path):
    """Total UMIs per cell from the file's own layout (CSR = by cell, CSC = by gene), read in chunks."""
    with h5py.File(path, 'r') as f:
        X = f['X']; n = int(X.attrs['shape'][0]); ip = X['indptr'][:]; out = np.zeros(n)
        if X.attrs.get('encoding-type') == 'csc_matrix':
            for a in range(0, int(ip[-1]), 5_000_000):
                b = min(a + 5_000_000, int(ip[-1]))
                out += np.bincount(X['indices'][a:b], weights=X['data'][a:b].astype(float), minlength=n)
        else:
            for s in range(0, n, 20_000):
                e = min(s + 20_000, n); a, b = int(ip[s]), int(ip[e])
                out[s:e] = np.bincount(np.repeat(np.arange(e - s), np.diff(ip[s:e + 1])), weights=X['data'][a:b].astype(float), minlength=e - s)
    return out

def label_kinds(labels):
    n = labels.map(lambda s: len([p for p in s.split(',') if p]))
    return pd.Series(np.select([labels.isin(['None', 'nan', '']), labels == 'CTRL', n == 1, n == 2],
                               ['no guide', 'control', 'single', 'pair'], '3+ guides'), index=labels.index)

def label_summary(name, min_cells=30, min_umi=1000):
    path = RAW[name][0]
    if not path.exists(): print(f'{name}: not on disk'); return None
    with h5py.File(path, 'r') as f: labels = read_elem(f['obs'])['gene'].astype(str).reset_index(drop=True)
    umi = cell_umis(path); kind = label_kinds(labels); real = umi >= min_umi
    print(f'== {name}: {len(labels):,} barcodes, {real.sum():,} with >= {min_umi:,} UMIs; median UMI {np.median(umi):,.0f}')
    display(pd.crosstab(kind, np.where(real, f'>= {min_umi} UMI', f'< {min_umi} UMI'), margins=True))
    singles = labels[(kind == 'single') & real].value_counts()
    print(f'single knockdowns in real cells: {len(singles)} genes, {int((singles >= min_cells).sum())} with >= {min_cells} cells; '
          f'controls: {int(((kind == "control") & real).sum()):,} cells')
    display(singles.rename('cells').to_frame().T)
    return dict(dataset=name, barcodes=len(labels), real_cells=int(real.sum()), controls=int(((kind == 'control') & real).sum()),
                single_cells=int(((kind == 'single') & real).sum()), knockdowns=len(singles), knockdowns_ge30=int((singles >= min_cells).sum()))

summary = [label_summary(n) for n in ['HFF', 'hESC']]
pd.DataFrame([s for s in summary if s]).set_index('dataset')

### 3c. Replogle K562 after cell QC
The QC (`scripts/qc_replogle_umi.py`, `scripts/qc_replogle_apply.py`; method in `results/qc/replogle_k562_gw/qc_method.md`) writes barcode lists, not a new file: the raw file is opened unchanged and `kept_cells.txt` selects the cells. Shown: cells before / after, removals by reason, the knockdowns that lost the most cells, the first 20 kept cells and their counts over the 20 genes with the most counts in them.

In [ ]:
QC = V / 'results/qc/replogle_k562_gw'
k562 = ad.read_h5ad(RAW['K562'][0], backed='r')
kept = pd.Index(pd.read_csv(QC / 'kept_cells.txt', header=None)[0].astype(str))
removed = pd.read_csv(QC / 'removed_cells.csv')
mask = k562.obs_names.isin(kept)
qc = k562[mask]                                      # a view on the raw file: nothing is copied or written
print(f'K562 genome-wide: {k562.n_obs:,} cells -> {qc.n_obs:,} kept ({qc.n_obs / k562.n_obs:.2%}), {len(removed):,} removed')
ctrl = k562.obs['gene'] == 'non-targeting'
print(f'controls: {ctrl.sum():,} -> {(ctrl & mask).sum():,}; knockdown cells: {(~ctrl).sum():,} -> {(~ctrl & mask).sum():,}; '
      f'knockdowns: {k562.obs.loc[~ctrl, "gene"].nunique():,} -> {k562.obs.loc[~ctrl & mask, "gene"].nunique():,}')
display(removed['reason'].value_counts().rename('cells removed').to_frame().T)
per = removed.groupby('gene').size().rename('removed').to_frame().join(k562.obs['gene'].value_counts().rename('cells'))
per['share'] = (per.removed / per.cells).round(3)
print('knockdowns losing the largest share of cells (>= 30 cells):')
display(per[per.cells >= 30].sort_values('share', ascending=False).head(10).T)
display(qc.obs.head(20))
block = np.asarray(k562.X[np.flatnonzero(mask)[:20]])
names = k562.var['gene_name'].astype(str).to_numpy(); top = np.argsort(-block.sum(0), kind='stable')[:20]
display(pd.DataFrame(block[:, top], index=qc.obs_names[:20], columns=names[top]))
k562.file.close()

### 3d. The saved QC-filtered files
Written by `scripts/write_replogle_qc.py --dataset <NAME>` to `lib/qc_cells/<source>_qc.h5ad` (method, cutoffs and checks in `results/qc/<source>/`): the kept cells only, counts as compressed sparse integers (identical values), `var` unchanged, every original obs column plus the standard ones (`source`, `background`, `condition`, `batch`, `target_gene`, `is_control`, `target_gene_id`, `guide_id`, `n_umi`, `qc_low_ref`). Shown per file: size, `uns['qc']` summary, the first 20 cells, the first 20 genes, a 20 × 20 block of counts, and the checks recorded when it was written.

In [ ]:
import json
QC_FILES = {'K562': (V / 'lib/qc_cells/replogle_k562_gw_qc.h5ad', V / 'results/qc/replogle_k562_gw'),
            'RPE1': (V / 'lib/qc_cells/replogle_rpe1_qc.h5ad', V / 'results/qc/replogle_rpe1'),
            'HepG2': (V / 'lib/qc_cells/nadig_hepg2_qc.h5ad', V / 'results/qc/nadig_hepg2')}
RAW['HepG2'] = (V / 'lib/downloads/nadig_hepg2/GSE264667_hepg2_raw_singlecell_01.h5ad', 'Nadig 2025')

def show_qc(name, n=20):
    path, qc_dir = QC_FILES[name]
    if not path.exists(): print(f'{name}: {path} not written yet'); return
    f = ad.read_h5ad(path, backed='r')
    print(f'{path.name}: {f.n_obs:,} cells x {f.n_vars:,} genes; {path.stat().st_size / 1e9:.1f} GB; counts {type(f.X).__name__}, {f.X.dtype}')
    print('uns["qc"] holds:', list(f.uns['qc'].keys()))
    display(f.obs.head(n)); display(f.var.head(n))
    block = f.X[:n].toarray()
    names = f.var['gene_name'].astype(str).to_numpy(); top = np.argsort(-block.sum(0), kind='stable')[:n]
    display(pd.DataFrame(block[:, top], index=f.obs_names[:n], columns=names[top]))
    q = f.uns['qc']
    print('counts:', dict(q['counts'])); print('lane minimum:', q['lane_min_controls'], '| whole-file cutoffs:', dict(q['whole_file_cutoffs']))
    print('checks (results/qc/.../checks.json):', json.loads((qc_dir / 'checks.json').read_text()))
    f.file.close()

show_qc('K562')

In [ ]:
show_qc('RPE1')

In [ ]:
show_qc('HepG2')

### 3e. Can the new lines serve as held-out validation lines?
For each new PerturBase source: knockdowns with ≥ 30 cells, whether each lowered its target by ≥ 50% (log2 fold change of mean CPM ≤ −1, target ≥ 5 CPM in controls; our rule), and how many passing knockdowns are also knocked down in our training lines (a held-out line can only test transfer for genes the model has seen elsewhere). Then the confounders: hESC replicates, HFF infection time course (only the uninfected 00h cells are kept).

Outcome (2026-10-18): MCF10A (0 of 29 knockdowns lower their target by ≥ 50%; it behaves like a Cas9 knockout, not CRISPRi) and LUHMES (5 of 14) were deleted. HFF was reduced to its uninfected 00h cells, knockdowns and controls (9,957 of 35,601 cells; `uns['subset']`); every 24–72 h cell was removed. hESC and HFF are the candidate held-out lines.

In [ ]:
PB_SOURCES = {'hESC (Genga 2019)': ['PRJNA524331'], 'HFF (Hein 2021)': ['CRISPRi_perturb_host']}
train = set()
for n in ['K562_gwps', 'RPE1', 'HepG2', 'Jurkat', 'HCT116', 'HEK293T', 'CD4_Stim48hr', 'H1']:
    train |= set(ad.read_h5ad(V / f'lib/filters/{n}.h5ad', backed='r').obs_names.astype(str))

def knockdown_table(adata, cells=None):
    """Per single knockdown: cells, control CPM of the target, on-target log2 fold change of mean CPM vs controls."""
    a = adata if cells is None else adata[cells]
    X = sp.csr_matrix(a.X); lab = a.obs['gene'].astype(str).to_numpy(); lib = np.asarray(X.sum(1)).ravel()
    ctrl = lab == 'CTRL'; single = ~ctrl & ~np.isin(lab, ['None', 'nan', '']) & ~pd.Series(lab).str.contains(',').to_numpy()
    cpm = sp.diags(1e6 / np.maximum(lib, 1)) @ X; pos = {g: i for i, g in enumerate(a.var_names.astype(str))}
    rows = []
    for t, n in pd.Series(lab[single]).value_counts().items():
        j = pos.get(t)
        c = float(cpm[ctrl][:, j].mean()) if j is not None else np.nan; k = float(cpm[lab == t][:, j].mean()) if j is not None else np.nan
        rows.append(dict(target=t, cells=n, ctrl_cpm=c, lfc=np.log2((k + 0.5) / (c + 0.5)) if j is not None else np.nan))
    r = pd.DataFrame(rows); r['passes'] = (r.cells >= 30) & (r.ctrl_cpm >= 5) & (r.lfc <= -1); r['in_training'] = r.target.isin(train)
    return r, int(ctrl.sum()), float(np.median(lib))

summary, passing = [], {}
for name, parts in PB_SOURCES.items():
    for part in parts:
        a = ad.read_h5ad(PB / f'{part}/raw.h5ad'); r, n_ctrl, med = knockdown_table(a)
        label = name if len(parts) == 1 else f'{name} {part}'; passing[label] = r[r.passes].sort_values('lfc')
        summary.append(dict(dataset=label, cells=a.n_obs, genes=a.n_vars, median_umi=med, controls=n_ctrl, knockdowns_ge30=int((r.cells >= 30).sum()),
                            pass_50pct=int(r.passes.sum()), median_lfc=round(float(r.loc[r.cells >= 30, 'lfc'].median()), 2),
                            pass_and_in_training=int((r.passes & r.in_training).sum())))
display(pd.DataFrame(summary).set_index('dataset'))
for k, r in passing.items(): print(f'{k}: {len(r)} passing:', ', '.join(r.target))

# confounders
hff = ad.read_h5ad(PB / 'CRISPRi_perturb_host/raw.h5ad')
print('HFF cells by infection time:', hff.obs['experimental_time'].astype(str).value_counts().to_dict())
r0, n0, _ = knockdown_table(hff, (hff.obs['experimental_time'].astype(str) == '00h').to_numpy())
print(f'HFF uninfected (00h) only: {n0} controls; knockdowns with >= 30 cells {int((r0.cells >= 30).sum())}, passing {int(r0.passes.sum())} '
      f'({int((r0.passes & r0.in_training).sum())} also in training); median cells per knockdown {int(r0.cells.median())}')
print('hESC replicates:', ad.read_h5ad(PB / 'PRJNA524331/raw.h5ad', backed='r').obs['batch'].astype(str).value_counts().to_dict())
